# RefMet Known Mapping Reconstruction

This notebook evaluates whether the saved final RefMet harmonization pipeline can reconstruct known lab mappings of:

`metabolite_name -> refmet_name`

It only uses rows where the lab file already contains both `metabolite_name` and `refmet_name`. It does **not** generate proposed mappings for currently unmapped lab names.

This is a reuse-only evaluation: no transformer embedding, no FAISS search, and no PubMed/PubChem fallback search are run here. The notebook consumes the saved outputs from `refmet_harmonization_publication_pipeline.ipynb`.

## Why This Evaluation Is Different

Earlier high MRR values around 0.70 came from easier or less conservative closed-set evaluations where exact/near-exact surface matches and reranked candidate pools made the task easier. This notebook is meant to answer the broader practical question:

> Across all known lab mappings, how much of the original `metabolite_name -> refmet_name` table can the current pipeline reconstruct?

The notebook reports both:

- **Full reconstruction**: the complete saved pipeline output.
- **Strict no-query-surface reconstruction**: a more conservative version using saved strict outputs where candidates that simply reproduce the query surface are removed.

## 1. Mount Google Drive

Run this in Colab. If you are running locally, the cell will skip Drive mounting.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', repr(exc))

## 2. Configure The Reconstruction Run

Leave `SOURCE_RUN_NAME` pointing at the completed final pipeline results folder. Set `WRITE_FULL_TOP10 = True` if you want large reviewer-ready top-10 candidate CSVs for all known mapped rows. Set it to `False` for a faster metrics-only run.

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/ChemSynonymizer/chem_synonymizer')
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd().resolve()

SOURCE_RUN_NAME = 'refmet_harmonization_publication_pipeline'
OUTPUT_NAME = 'known_mapping_reconstruction'
RANDOM_SEED = 7
SAMPLE_PER_CASE_GROUP = 150
WRITE_FULL_TOP10 = True

env_values = {
    'CHEMSYN_PROJECT_DIR': str(PROJECT_DIR),
    'CHEMSYN_SOURCE_RUN_NAME': SOURCE_RUN_NAME,
    'CHEMSYN_OUTPUT_NAME': OUTPUT_NAME,
    'CHEMSYN_RANDOM_SEED': str(RANDOM_SEED),
    'CHEMSYN_SAMPLE_PER_CASE_GROUP': str(SAMPLE_PER_CASE_GROUP),
    'CHEMSYN_WRITE_FULL_TOP10': str(WRITE_FULL_TOP10),
}
for key, value in env_values.items():
    os.environ[key] = value

print('Project:', PROJECT_DIR)
print('Source results:', PROJECT_DIR / 'results' / SOURCE_RUN_NAME)
print('Output:', PROJECT_DIR / 'results' / SOURCE_RUN_NAME / OUTPUT_NAME)
print('Write full top-10 candidate files:', WRITE_FULL_TOP10)

## 3. Verify Required Saved Outputs

In [ ]:
source_results = PROJECT_DIR / 'results' / SOURCE_RUN_NAME
required = [
    source_results / 'refmet_predictions_with_decisions.csv',
    source_results / 'refmet_predictions_strict_no_query_surface_with_decisions.csv',
    source_results / 'refmet_top10_candidates_wide.csv',
    source_results / 'refmet_top10_candidates_wide_strict_no_query_surface.csv',
]

for path in required:
    status = 'OK' if path.exists() else 'MISSING'
    size = path.stat().st_size / (1024**2) if path.exists() else None
    print(f'{status:7} {path}' + (f' ({size:.1f} MB)' if size is not None else ''))

missing = [p for p in required[:2] if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required prediction files: ' + ', '.join(str(p) for p in missing))
if WRITE_FULL_TOP10:
    missing_top10 = [p for p in required[2:] if not p.exists()]
    if missing_top10:
        raise FileNotFoundError('WRITE_FULL_TOP10=True but top-10 files are missing: ' + ', '.join(str(p) for p in missing_top10))

## 4. Run Reconstruction Evaluation

This should be much faster than the original pipeline. The only potentially slow part is writing the full top-10 candidate review files if `WRITE_FULL_TOP10=True`.

In [ ]:
import subprocess
import sys

candidates = [
    PROJECT_DIR / 'scripts' / 'run_refmet_known_mapping_reconstruction.py',
    Path.cwd() / 'scripts' / 'run_refmet_known_mapping_reconstruction.py',
    Path.cwd().parent / 'scripts' / 'run_refmet_known_mapping_reconstruction.py',
]
script_path = next((p for p in candidates if p.exists()), None)
if script_path is None:
    raise FileNotFoundError('Could not find scripts/run_refmet_known_mapping_reconstruction.py')

print('Running:', script_path)
cmd = [sys.executable, str(script_path), '--project-dir', str(PROJECT_DIR)]
if not WRITE_FULL_TOP10:
    cmd.append('--no-write-full-top10')
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
return_code = proc.wait()
if return_code != 0:
    raise RuntimeError(f'Known mapping reconstruction failed with exit code {return_code}')

## 5. Inspect Metrics

Use `all_splits` to answer the full reconstruction question. Use `test` when you want a held-out estimate. Use `nonidentical_metabolite_refmet` and the strict outputs for the most conservative interpretation.

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown

output_dir = PROJECT_DIR / 'results' / SOURCE_RUN_NAME / OUTPUT_NAME
print('Output directory:', output_dir)

dataset_path = output_dir / 'known_mapping_dataset_summary.json'
metrics_path = output_dir / 'known_mapping_reconstruction_metrics.csv'
acceptance_path = output_dir / 'known_mapping_acceptance_summary.csv'
summary_path = output_dir / 'known_mapping_reconstruction_summary.md'

if dataset_path.exists():
    print(json.dumps(json.loads(dataset_path.read_text()), indent=2))

metrics = pd.read_csv(metrics_path)
display(metrics[(metrics['split'] == 'all_splits')].sort_values(['variant', 'subset']))

acceptance = pd.read_csv(acceptance_path)
display(acceptance[(acceptance['split'] == 'all_splits')].sort_values(['variant', 'decision_bucket']))

if summary_path.exists():
    display(Markdown(summary_path.read_text()))

## 6. Inspect Case Groups And Human Review Examples

In [ ]:
case_path = output_dir / 'known_mapping_case_group_summary.csv'
case_summary = pd.read_csv(case_path)
display(case_summary[(case_summary['split'] == 'all_splits')].sort_values(['variant', 'case_group']))

examples_path = output_dir / 'strict_no_query_surface_known_mapping_examples.csv'
if examples_path.exists():
    examples = pd.read_csv(examples_path)
    print(examples['example_group'].value_counts(dropna=False))
    display(examples.head(30))

## 7. Plot Summary

In [ ]:
from IPython.display import Image, display

plots_dir = output_dir / 'plots'
for name in [
    'known_mapping_reconstruction_MRR.png',
    'known_mapping_reconstruction_Hitat1.png',
    'known_mapping_reconstruction_Hitat10.png',
    'known_mapping_reconstruction_Hitat256.png',
]:
    path = plots_dir / name
    if path.exists():
        display(Image(filename=str(path)))